# Building Decision Trees

## Goal

Build a decision tree from scratch to understand how tree-based machine learning models learn rules, evaluate splits, and make predictions.

The project starts from the simplest possible model and evolves step by step toward numerical splits, deeper trees, and boosting.

---

## Version 3

Turn the mixed decision stump into a recursive decision tree.

The model will:

- reuse categorical and numerical split search
- allow each branch to create another decision node
- build multiple levels recursively
- create terminal leaves when splitting should stop
- limit tree growth using a maximum depth
- route observations through multiple learned decisions
- generate a new Titanic submission

In [1]:
import pandas as pd

In [2]:
PATH = "/kaggle/input/competitions/titanic/"

train = pd.read_csv(PATH + "train.csv", index_col="PassengerId")
test  = pd.read_csv(PATH + "test.csv", index_col="PassengerId")

In [3]:
train.head()

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
PassengerId,,,,,,,,,,,
1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [4]:
# Feature engineering

def add_title_feature(df):
    df = df.copy()

    names = df["Name"].str.lower().str.strip()
    df["Title"] = names.str.extract(r" ([a-z]+)\.", expand=False)

    return df

train = add_title_feature(train)
test = add_title_feature(test)

In [5]:
# Define target and feature types

TARGET = "Survived"

NUMERIC_FEATURES = ["Age", "Fare"]
CATEGORICAL_FEATURES = ["Sex", "Title", "Pclass"]

FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

In [6]:
# Check missing values before imputation

print("Train missing values:")
print(train[FEATURES].isna().sum())

print("\nTest missing values:")
print(test[FEATURES].isna().sum())

Train missing values:
Age       177
Fare        0
Sex         0
Title       0
Pclass      0
dtype: int64

Test missing values:
Age       86
Fare       1
Sex        0
Title      0
Pclass     0
dtype: int64


In [7]:
# Fill missing numerical values using training medians

age_median = train["Age"].median()
fare_median = train["Fare"].median()

train["Age"] = train["Age"].fillna(age_median)
test["Age"] = test["Age"].fillna(age_median)

test["Fare"] = test["Fare"].fillna(fare_median)

## Data preparation

We create the training features, the target, and the test features used by the model.

In [8]:
# Prepare training and test features

X_train = train[FEATURES]
y_train = train[TARGET]

X_test = test[FEATURES]

In [9]:
X_train.head()

,Age,Fare,Sex,Title,Pclass
PassengerId,,,,,
1,22.0,7.2500,male,mr,3
2,38.0,71.2833,female,mrs,1
3,26.0,7.9250,female,miss,3
4,35.0,53.1000,female,mrs,1
5,35.0,8.0500,male,mr,3


## Building the model

We now extend the decision stump into a recursive decision tree.

Each branch can become a new decision node until a stopping condition creates a leaf.

In [10]:
# Fit a constant predictor

def fit_constant(y):
    if len(y) == 0:
        raise ValueError("Target cannot be empty.")

    return sum(y) / len(y)

In [11]:
# Compute mean squared error

def mean_squared_error(y, prediction):
    if len(y) == 0:
        raise ValueError("Target cannot be empty.")

    total_error = 0

    for value in y:
        total_error += (value - prediction) ** 2

    return total_error / len(y)

In [12]:
# Split target values using a categorical equality rule

def split_categorical(x, y, category):
    y_left = []
    y_right = []

    for feature_value, target_value in zip(x, y):
        if feature_value == category:
            y_left.append(target_value)
        else:
            y_right.append(target_value)

    return y_left, y_right

In [13]:
# Split target values using a numerical threshold

def split_numeric(x, y, threshold):
    y_left = []
    y_right = []

    for feature_value, target_value in zip(x, y):
        if feature_value <= threshold:
            y_left.append(target_value)
        else:
            y_right.append(target_value)

    return y_left, y_right

### Recursive dataset splitting

A decision stump only needs the target values in each branch.

A recursive tree also needs the feature rows that reach each branch, because every branch may search for another split.

In [14]:
# Split the full dataset using a learned rule

def split_dataset(X, y, feature, split_type, split_value):
    if split_type == "categorical":
        go_left = X[feature] == split_value
    else:
        go_left = X[feature] <= split_value

    X_left = X[go_left]
    y_left = y[go_left]

    X_right = X[~go_left]
    y_right = y[~go_left]

    return X_left, y_left, X_right, y_right

### Leaf nodes

A leaf is a terminal node that no longer splits the data.

It stores the prediction learned from the target values that reach it.

In [15]:
# Create a leaf node

def make_leaf(y):
    prediction = fit_constant(y)

    return {
        "type": "leaf",
        "prediction": prediction
    }

### Stopping conditions

A recursive tree cannot split forever.

A branch becomes a leaf when the target is already pure, when the maximum depth is reached, or when no useful split can be found.

In [16]:
# Check whether a branch should stop growing

def should_stop(y, depth, max_depth):
    if depth >= max_depth:
        return True

    if len(set(y)) == 1:
        return True

    return False

In [17]:
# Compute the weighted MSE after a categorical split

def categorical_split_mse(x, y, category):
    y_left, y_right = split_categorical(x, y, category)

    left_prediction = fit_constant(y_left)
    right_prediction = fit_constant(y_right)

    left_mse = mean_squared_error(y_left, left_prediction)
    right_mse = mean_squared_error(y_right, right_prediction)

    n_left = len(y_left)
    n_right = len(y_right)

    return (n_left * left_mse + n_right * right_mse) / (n_left + n_right)

In [18]:
# Compute the weighted MSE after a numerical split

def numeric_split_mse(x, y, threshold):
    y_left, y_right = split_numeric(x, y, threshold)

    left_prediction = fit_constant(y_left)
    right_prediction = fit_constant(y_right)

    left_mse = mean_squared_error(y_left, left_prediction)
    right_mse = mean_squared_error(y_right, right_prediction)

    n_left = len(y_left)
    n_right = len(y_right)

    return (n_left * left_mse + n_right * right_mse) / (n_left + n_right)

In [19]:
# Compute the error reduction produced by a categorical split

def categorical_split_gain(x, y, category):
    base_prediction = fit_constant(y)
    base_mse = mean_squared_error(y, base_prediction)

    split_mse = categorical_split_mse(x, y, category)

    return base_mse - split_mse

In [20]:
# Compute the error reduction produced by a numerical split

def numeric_split_gain(x, y, threshold):
    base_prediction = fit_constant(y)
    base_mse = mean_squared_error(y, base_prediction)

    split_mse = numeric_split_mse(x, y, threshold)

    return base_mse - split_mse

### Numerical split search

For numerical features, the model must find the threshold that produces the largest gain.

Instead of testing every possible number, we sort the unique values and test the midpoints between consecutive values.

In [21]:
# Find the best numerical split for one feature

def find_best_numeric_split(x, y):
    values = []

    for value in x:
        if value not in values:
            values.append(value)

    values.sort()

    if len(values) < 2:
        return None, float("-inf")

    thresholds = []

    for i in range(len(values) - 1):
        midpoint = (values[i] + values[i + 1]) / 2
        thresholds.append(midpoint)

    best_threshold = None
    best_gain = float("-inf")

    for threshold in thresholds:
        gain = numeric_split_gain(x, y, threshold)

        if gain > best_gain:
            best_threshold = threshold
            best_gain = gain

    return best_threshold, best_gain

In [22]:
# Find the best categorical split for one feature

def find_best_categorical_split(x, y):
    categories = []

    for value in x:
        if value not in categories:
            categories.append(value)

    if len(categories) < 2:
        return None, float("-inf")

    best_category = None
    best_gain = float("-inf")

    for category in categories:
        gain = categorical_split_gain(x, y, category)

        if gain > best_gain:
            best_category = category
            best_gain = gain

    return best_category, best_gain

In [23]:
# Find the best split across all features

def find_best_feature_split(X, y):
    best_feature = None
    best_split_type = None
    best_split_value = None
    best_gain = float("-inf")

    for feature in X:
        if feature in NUMERIC_FEATURES:
            split_value, gain = find_best_numeric_split(X[feature], y)
            split_type = "numeric"
        else:
            split_value, gain = find_best_categorical_split(X[feature], y)
            split_type = "categorical"

        if gain > best_gain:
            best_feature = feature
            best_split_type = split_type
            best_split_value = split_value
            best_gain = gain

    return best_feature, best_split_type, best_split_value, best_gain

In [24]:
# Build a recursive decision tree

def build_tree(X, y, depth=0, max_depth=3):
    if should_stop(y, depth, max_depth):
        return make_leaf(y)

    best_feature, best_split_type, best_split_value, best_gain = find_best_feature_split(X, y)

    if best_gain <= 0:
        return make_leaf(y)

    X_left, y_left, X_right, y_right = split_dataset(
        X,
        y,
        best_feature,
        best_split_type,
        best_split_value
    )

    left_child = build_tree(X_left, y_left, depth + 1, max_depth)
    right_child = build_tree(X_right, y_right, depth + 1, max_depth)

    return {
        "type": "node",
        "feature": best_feature,
        "split_type": best_split_type,
        "split_value": best_split_value,
        "gain": best_gain,
        "left": left_child,
        "right": right_child
    }

### Train the recursive tree

The model now builds multiple decision nodes recursively.

Each branch can become another node until a stopping condition creates a leaf.

In [25]:
# Train the Version 3 recursive tree

tree = build_tree(X_train, y_train, max_depth=3)
tree

{'type': 'node',
 'feature': 'Title',
 'split_type': 'categorical',
 'split_value': 'mr',
 'gain': 0.07133502379453835,
 'left': {'type': 'node',
  'feature': 'Fare',
  'split_type': 'numeric',
  'split_value': 26.26875,
  'gain': 0.00986313643290919,
  'left': {'type': 'node',
   'feature': 'Age',
   'split_type': 'numeric',
   'split_value': 32.5,
   'gain': 0.0005279600124964146,
   'left': {'type': 'leaf', 'prediction': 0.1134020618556701},
   'right': {'type': 'leaf', 'prediction': 0.06060606060606061}},
  'right': {'type': 'node',
   'feature': 'Fare',
   'split_type': 'numeric',
   'split_value': 26.775,
   'gain': 0.01834955809551264,
   'left': {'type': 'leaf', 'prediction': 0.6875},
   'right': {'type': 'leaf', 'prediction': 0.27927927927927926}}},
 'right': {'type': 'node',
  'feature': 'Pclass',
  'split_type': 'categorical',
  'split_value': 3,
  'gain': 0.039470854277977224,
  'left': {'type': 'node',
   'feature': 'Fare',
   'split_type': 'numeric',
   'split_value': 23.

### Learned tree structure

The trained model is now a recursive decision tree.

The root still starts with `Title == "mr"`, but each branch can continue with new decisions such as `Fare`, `Age`, or `Pclass`.

Each internal node stores a learned rule, while each leaf stores the target average of the samples that reached it.

The same feature can appear more than once because every node works on a different subset of the data.

### Recursive prediction

To make a prediction, one observation starts at the root and follows the learned rules.

At each node, it moves left or right until it reaches a leaf.

The leaf value becomes the final prediction.

In [26]:
# Predict one observation by traversing the tree

def predict_one(row, tree):
    if tree["type"] == "leaf":
        return tree["prediction"]

    feature_value = row[tree["feature"]]

    if tree["split_type"] == "categorical":
        go_left = feature_value == tree["split_value"]
    else:
        go_left = feature_value <= tree["split_value"]

    if go_left:
        return predict_one(row, tree["left"])
    else:
        return predict_one(row, tree["right"])

## Prediction and submission

The recursive tree returns the value stored in the leaf reached by each observation.

We convert these leaf values into binary classes and generate the Version 3 Kaggle submission.

In [27]:
# Predict leaf values for all observations

def predict_proba_tree(X, tree):
    predictions = []

    for _, row in X.iterrows():
        predictions.append(predict_one(row, tree))

    return predictions

In [28]:
# Convert leaf predictions into binary classes

def predict_tree(X, tree, threshold=0.5):
    probabilities = predict_proba_tree(X, tree)
    predictions = []

    for probability in probabilities:
        if probability >= threshold:
            predictions.append(1)
        else:
            predictions.append(0)

    return predictions

In [29]:
# Generate binary predictions for the Kaggle test set

test_predictions = predict_tree(X_test, tree)

In [30]:
submission = pd.DataFrame({
    "PassengerId": test.index,
    "Survived": test_predictions
})

submission.to_csv("submission_v3.csv", index=False)

submission.head()

,PassengerId,Survived
0,892,0
1,893,1
2,894,0
3,895,0
4,896,1


## Notes

This third version turns the mixed decision stump into a recursive decision tree.

- Each branch can become another decision node.
- Every node searches independently for the best categorical or numerical split.
- The same feature can appear multiple times with different split values.
- Leaves store the target average of the samples that reach them.
- Tree growth stops when a node is pure, no useful split exists, or the maximum depth is reached.
- Prediction recursively follows the learned rules until a leaf is reached.
- Missing numerical values are still filled using medians computed from the training set.
- Final Titanic predictions are converted into binary classes (`0` or `1`).

Future versions will introduce new components and gradually evolve the architecture.